# Metop ASCAT soil moisture

Fetch a Metop ASCAT Level 2 soil-moisture product (12.5 km) — a BUFR swath product over a land region.

This notebook performs a **real** EUMETSAT Data Store download when `EUMETSAT_CONSUMER_KEY` / `EUMETSAT_CONSUMER_SECRET` are set (see the [authentication guide](../../reference/eumetsat/authentication.md)). Without credentials it prints a `skipped live query` line and still runs to the end.

## Setup

Import the unified `EarthLens` entry point and prepare a local output directory for the downloaded product.

In [ ]:
from pathlib import Path

from earthlens import EarthLens

OUT_DIR = Path('eumetsat_output')
OUT_DIR.mkdir(exist_ok=True)

## Build the request

Describe the EUMETSAT query — the ASCAT 12.5 km soil-moisture dataset, the `SOMO12` variable, a single day, and a bounding box over Iberia / north-west Africa. The constructor only assembles the request; nothing is fetched yet.

In [ ]:
client = EarthLens(
    data_source='eumetsat',
    dataset='metop-ascat-soil-moisture-12km',
    variables=['SOMO12'],
    start='2024-06-01',
    end='2024-06-01',
    aoi=[-5.0, 35.0, 5.0, 45.0],
    path=str(OUT_DIR),
)

## Download the product

Run the live download. The whole call is wrapped in a `try`/`except` so the notebook still runs end-to-end without `EUMETSAT_CONSUMER_KEY` / `EUMETSAT_CONSUMER_SECRET` (or during a transient Data Store hiccup): it simply reports a `skipped live query` line instead.

In [ ]:
paths = None
try:
    paths = client.download(progress_bar=False)
    print(len(paths), 'product(s):', [Path(p).name for p in paths])
except Exception as exc:
    # No EUMETSAT_CONSUMER_KEY/SECRET, or a transient Data Store hiccup:
    # the notebook still runs top-to-bottom and reports the skip.
    print(f'skipped live query: {type(exc).__name__}: {exc}')

ASCAT soil moisture is delivered as BUFR; decode it with an external BUFR reader (e.g. ecCodes) after download.